# Capítulo 8 · Delegar con aislamiento

Tres tareas que tocan los mismos archivos, para tres agentes a la vez. Se comparan cuatro formas de repartirlas: cada una en su copia y gana la última, todas en la misma copia, en oleadas y cada una en su copia con un integrador que fusiona a tres bandas y solo deja entrar lo que deja las pruebas en verde. El agente es el del capítulo 7, con marco, plan y revisor.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. Las secciones 7 y 8 se pueden hacer sin GPU.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo, el repositorio y las herramientas

In [ ]:
!pip install -q transformers==5.19.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 8: delegar con aislamiento."""
import ast
import importlib
import io
import json
import re
import subprocess
import sys
import tempfile
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
RAIZ = Path.cwd()                           # donde están datos/ y comun_libro/
REPOSITORIO = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
               "archive/refs/heads/main.zip")
TAREAS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
          "T17", "T18", "T19", "T20"]       # las tareas de una sola sesión
MAX_PASOS = 30             # llamadas al modelo por tarea
MAX_TOKENS_PASO = 600      # tokens nuevos por llamada
LIMITE_ENTRADA = 12_000    # tokens de la conversación: más no cabe en la T4
RESULTADOS_INTACTOS = 2    # los últimos resultados de herramientas no se recortan
MAX_RECHAZOS = 3           # veces que la puerta devuelve el trabajo
VORAZ = {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}
MUESTREO = {"do_sample": True, "temperature": 0.7, "top_p": 0.8,
            "top_k": 20}           # los valores que recomienda Qwen para este modelo
decodificacion = dict(VORAZ)       # la que usa generar(); probar() la cambia
hr = None                  # el módulo de herramientas, tras preparar()



def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def preparar():
    """El repositorio de Ciclos Norte, sus tareas y las herramientas del agente.
    Si no están (en Colab), se descargan del repositorio del libro."""
    global hr
    if not (RAIZ / "comun_libro" / "herramientas_repo.py").exists():
        with urllib.request.urlopen(REPOSITORIO) as respuesta:
            archivo = zipfile.ZipFile(io.BytesIO(respuesta.read()))
        for nombre in archivo.namelist():
            relativa = nombre.split("/", 1)[1]
            if relativa.startswith(("datos/", "comun_libro/")) and not nombre.endswith("/"):
                destino = RAIZ / relativa
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(archivo.read(nombre))
    sys.path.insert(0, str(RAIZ / "comun_libro"))
    hr = importlib.import_module("herramientas_repo")
    print(f"Repositorio en {hr.REPO}: {len(hr.archivos_de(hr.REPO))} archivos")

In [ ]:
tok, modelo = cargar()
preparar()

## 2. El modelo

In [ ]:
def plantilla(tok, mensajes):
    """La conversación como texto, con las herramientas, en la plantilla del modelo."""
    return tok.apply_chat_template(mensajes, tools=hr.ESQUEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generar(tok, modelo, texto):
    """Una respuesta del modelo, con la decodificación del momento: voraz o con
    muestreo. Devuelve el texto y los tokens de entrada y de salida."""
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, **decodificacion, max_new_tokens=MAX_TOKENS_PASO,
                                 prefill_chunk_size=1024, pad_token_id=tok.eos_token_id)
    n_entrada = entrada["input_ids"].shape[1]
    nuevos = salida[0, n_entrada:].cpu()
    del entrada, salida
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # cada paso tiene otra longitud: sin esto la
        #                           memoria reservada crece hasta llenar la GPU
    return tok.decode(nuevos, skip_special_tokens=True).strip(), n_entrada, len(nuevos)


def leer_llamadas(respuesta):
    """Las llamadas {name, arguments} que el modelo escribe entre <tool_call> y
    </tool_call>. Una llamada que no se entiende queda con name None."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", respuesta, re.S):
        try:
            datos = json.loads(bloque.strip(), strict=False)
        except ValueError:
            datos = None
        if isinstance(datos, dict) and "name" in datos:
            argumentos = datos.get("arguments", {})
            llamadas.append({"name": datos["name"],
                             "arguments": argumentos if isinstance(argumentos, dict)
                             else {}})
        else:
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def recortar_conversacion(tok, mensajes):
    """Si la conversación pasa de LIMITE_ENTRADA tokens, cambia los resultados de
    herramientas más antiguos por un aviso corto. Devuelve el texto para el modelo y
    cuántos ha recortado (None si ni así cabe)."""
    texto = plantilla(tok, mensajes)
    recortados = 0
    candidatos = [m for m in mensajes if m["role"] == "tool"
                  and not m.get("recortado")][:-RESULTADOS_INTACTOS or None]
    while len(tok(texto)["input_ids"]) > LIMITE_ENTRADA:
        if not candidatos:
            return texto, None
        mensaje = candidatos.pop(0)
        mensaje["content"] = json.dumps(
            {"aviso": "Resultado antiguo recortado para ahorrar memoria; vuelve a "
                      "pedirlo si lo necesitas.",
             "inicio": mensaje["content"][:150]}, ensure_ascii=False)
        mensaje["recortado"] = True
        recortados += 1
        texto = plantilla(tok, mensajes)
    return texto, recortados

## 3. Las reglas del capítulo 2 y el contrato del capítulo 4

In [ ]:
SISTEMA = (
    "Eres un agente de programación que trabaja en el repositorio de Ciclos Norte, "
    "una tienda de bicicletas de Oviedo. Es un programa de gestión en Python con "
    "varios módulos (pedidos, devoluciones, existencias, facturas, avisos…), sus "
    "pruebas en la carpeta tests/ y unos datos de ejemplo. Recibirás una petición de "
    "alguien de la tienda y tienes que resolverla cambiando el código con las "
    "herramientas: listar archivos, leer por rangos de líneas, buscar, editar por "
    "sustitución exacta, ejecutar las pruebas y unas pocas órdenes permitidas. "
    "Trabaja paso a paso, con una herramienta cada vez. Cuando hayas terminado, "
    "responde sin llamar a ninguna herramienta, empezando por «TERMINADO:» y "
    "explicando en una o dos frases qué has cambiado. Si no puedes terminar, empieza "
    "por «NO TERMINADO:» y explica por qué.")


AGENTS_REGLAS = """# AGENTS.md · Ciclos Norte

Reglas del repositorio. Cada una se comprueba al final de tu trabajo.

R1. Antes de editar un archivo, léelo con leer_archivo.
R2. No cambies nada de datos/.
R3. Cambia docs/ solo si la petición pide cambiar las condiciones generales.
R4. No borres ni cambies pruebas que ya existen. Excepción: si la petición cambia una
    regla del negocio, actualiza las pruebas de la regla antigua y dilo en tu respuesta.
R5. Añade tus pruebas en el archivo del módulo (tests/test_envios.py para envios.py):
    sustituye la línea `if __name__ == "__main__":` por una clase nueva con tu prueba,
    seguida de esa misma línea. Importa dentro de la prueba lo que necesites.
R6. No crees módulos nuevos fuera de tests/.
R7. «TERMINADO» exige: cambios en el código del programa, una comprobación nueva
    en tests/ y ejecutar_pruebas en verde después de tu último cambio.
R8. Tu respuesta final nombra todos los archivos que has cambiado.
"""


PIDE_DOCS = re.compile(r"(actualiz\w*|y en las)\s[^.]{0,40}condiciones\s+generales", re.I)


def lineas_del_diff(diff):
    """Las líneas añadidas y quitadas de cada archivo de un diff unificado:
    {ruta: {"+": [...], "-": [...]}}."""
    archivos, actual = {}, None
    for linea in diff.splitlines():
        if linea.startswith("+++ b/"):
            actual = archivos.setdefault(linea[6:], {"+": [], "-": []})
        elif linea.startswith("--- a/") or actual is None:
            continue
        elif linea[:1] in "+-" and linea[1:].strip():
            actual[linea[0]].append(linea[1:])
    return archivos


def comprobar_reglas(tarea, registro, cambios, respuesta, exigir_prueba=True,
                     permite_docs=None):
    """Las reglas incumplidas, como {regla: evidencia}. Mira lo que de verdad ha
    cambiado (el diff), el registro de llamadas y la respuesta final. Con
    «exigir_prueba» en False, la R7 no pide una comprobación nueva; «permite_docs»,
    si se da, decide la R3 en lugar de la petición."""
    if permite_docs is None:
        permite_docs = bool(PIDE_DOCS.search(tarea["peticion"]))
    rotas = {}
    lineas = lineas_del_diff(cambios["diff"])
    leidos = set()
    for r in registro:
        ruta = r["argumentos"].get("ruta", "")
        if r["herramienta"] == "leer_archivo" and r["ok"]:
            leidos.add(ruta)
        elif r["herramienta"] == "editar" and r["ok"] and ruta not in leidos:
            rotas.setdefault("R1", f"editaste {ruta} sin leerlo antes")
    for ruta in cambios["cambiados"]:
        if ruta.startswith("datos/"):
            rotas["R2"] = f"cambiaste {ruta}"
        if ruta.startswith("docs/") and not permite_docs:
            rotas["R3"] = f"cambiaste {ruta} y la petición no lo pide"
        if ruta.startswith("tests/") and ruta in cambios["nuevos"]:
            rotas["R5"] = f"creaste {ruta}; las pruebas van en el archivo del módulo"
        if not ruta.startswith(("tests/", "docs/", "datos/")) and ruta in cambios["nuevos"]:
            rotas["R6"] = f"creaste el módulo {ruta}"
    quitadas = [r for r, l in lineas.items() if r.startswith("tests/") and l["-"]]
    if quitadas and not re.search(r"prueba", respuesta or "", re.I):
        rotas["R4"] = (f"cambiaste pruebas que ya existían en {', '.join(quitadas)} "
                       "y tu respuesta no lo dice")
    if respuesta and dice_terminado(respuesta):
        programa = [r for r in cambios["cambiados"]
                    if not r.startswith(("tests/", "docs/", "datos/"))]
        prueba_nueva = any(re.search(r"def test_|assert", l)
                           for r, l2 in lineas.items() if r.startswith("tests/")
                           for l in l2["+"])
        ediciones = [i for i, r in enumerate(registro)
                     if r["herramienta"] in ("editar", "crear_archivo") and r["ok"]]
        despues = registro[ediciones[-1] + 1:] if ediciones else registro
        pruebas = [r for r in despues if r["herramienta"] == "ejecutar_pruebas"]
        falta = [texto for cumple, texto in (
            (programa, "no hay cambios en el código del programa"),
            (prueba_nueva or not exigir_prueba,
             "no hay ninguna comprobación nueva en tests/"),
            (pruebas and pruebas[-1]["resultado"].get("todas_pasan"),
             "no hay una ejecución de las pruebas en verde después del último cambio"))
            if not cumple]
        if falta:
            rotas["R7"] = "; ".join(falta)
        sin_nombrar = [r for r in cambios["cambiados"]
                       if Path(r).name not in respuesta]
        if sin_nombrar:
            rotas["R8"] = f"tu respuesta no nombra {', '.join(sin_nombrar)}"
    return rotas


def dice_terminado(respuesta):
    """True si la respuesta final dice que la tarea está hecha. Una respuesta vacía
    no dice nada."""
    if not respuesta.strip():
        return False
    if re.search(r"NO\s+TERMINADO", respuesta, re.I):
        return False
    if re.search(r"TERMINADO", respuesta):
        return True
    return not re.search(r"no (he podido|puedo|consigo|es posible)|necesito|\?",
                         respuesta, re.I)


class HerramientasConReglas:
    """Las herramientas del agente con las reglas de proceso comprobadas antes de
    cada edición: si la edición incumple R1, R2, R3, R5 o R6, no se hace y el agente
    recibe la regla. El resto de llamadas pasan tal cual."""

    def __init__(self, herramientas, tarea):
        self.herramientas = herramientas
        self.registro = herramientas.registro
        self.tarea = tarea
        self.leidos = set()

    def regla_rota(self, nombre, ruta):
        """La regla que incumpliría editar o crear «ruta», o None."""
        existe = self.herramientas.espacio.ruta(ruta).exists()
        if ruta.startswith("datos/"):
            return "R2: no cambies nada de datos/."
        if ruta.startswith("docs/") and not self.permite_docs():
            return "R3: la petición no pide cambiar las condiciones generales."
        if nombre == "crear_archivo" and ruta.startswith("tests/"):
            return ("R5: no crees archivos de pruebas; añade tu prueba al archivo del "
                    "módulo, sustituyendo la línea if __name__ == \"__main__\":")
        if nombre == "crear_archivo" and not existe:
            return "R6: no crees módulos nuevos fuera de tests/."
        if nombre == "editar" and existe and ruta not in self.leidos:
            return f"R1: lee {ruta} con leer_archivo antes de editarlo."
        return None

    def permite_docs(self):
        """True si la petición pide cambiar las condiciones generales (R3)."""
        return bool(PIDE_DOCS.search(self.tarea["peticion"]))

    def llamar(self, nombre, argumentos):
        """Como Herramientas.llamar, pero comprobando las reglas antes de editar."""
        ruta = str((argumentos or {}).get("ruta", ""))
        if nombre in ("editar", "crear_archivo"):
            regla = self.regla_rota(nombre, ruta)
            if regla:
                resultado = {"error": "Edición rechazada por AGENTS.md. " + regla}
                self.registro.append({"herramienta": nombre, "argumentos": argumentos,
                                      "ok": False, "segundos": 0,
                                      "resultado": resultado})
                return resultado
        resultado = self.herramientas.llamar(nombre, argumentos)
        if nombre == "leer_archivo" and "error" not in resultado:
            self.leidos.add(ruta)
        return resultado


def contrato_texto(alcance):
    """Un contrato (como datos/tareas/Txx/alcance.json) escrito para el agente:
    objetivo, rutas que puede cambiar, rutas prohibidas y criterios de aceptación
    numerados (C1, C2…)."""
    criterios = "\n".join(f"C{i}. {c}" for i, c in
                          enumerate(alcance["criterios_aceptacion"], 1))
    return ("Contrato de la tarea\n"
            f"Objetivo: {alcance['objetivo']}\n"
            f"Puedes cambiar: {', '.join(alcance['rutas_permitidas'])}\n"
            f"No toques: {', '.join(alcance['rutas_prohibidas'])}\n"
            f"Criterios de aceptación:\n{criterios}\n"
            "El contrato manda sobre AGENTS.md: si prohíbe tests/, no añadas pruebas.")


def prohibe_pruebas(alcance):
    """True si el contrato prohíbe cambiar tests/ (por ejemplo, una refactorización
    que debe pasar las pruebas sin tocarlas)."""
    return hr.coincide("tests/x.py", alcance["rutas_prohibidas"])


def comprobar_contrato(alcance, cambios, respuesta):
    """Lo que incumple el contrato al terminar: archivos fuera de las rutas permitidas
    o en las prohibidas, y criterios que la respuesta no explica cómo se cumplen."""
    rotas = {}
    rutas = hr.comprobar_alcance(cambios["cambiados"], alcance)
    fuera = sorted(set(rutas["fuera_de_alcance"]) | set(rutas["prohibidos"]))
    if fuera:
        rotas["Alcance"] = f"cambiaste {', '.join(fuera)}, que el contrato no permite"
    numeros = range(1, len(alcance["criterios_aceptacion"]) + 1)
    sin_explicar = [f"C{i}" for i in numeros
                    if not re.search(rf"\bC{i}\b", respuesta or "")]
    if sin_explicar:
        rotas["Criterios"] = ("tu respuesta no dice cómo se cumple "
                              + ", ".join(sin_explicar)
                              + " (cita cada criterio por su número)")
    return rotas


class HerramientasConContrato(HerramientasConReglas):
    """Las herramientas con las reglas del capítulo 2 y, además, el contrato: no se
    puede editar ni crear nada fuera de las rutas permitidas."""

    def __init__(self, herramientas, tarea, alcance):
        super().__init__(herramientas, tarea)
        self.alcance = alcance

    def permite_docs(self):
        """El contrato manda sobre la R3: docs/ se puede cambiar si lo permite."""
        return hr.coincide("docs/condiciones_generales.md", self.alcance["rutas_permitidas"])

    def regla_rota(self, nombre, ruta):
        """La regla o la parte del contrato que incumpliría la edición, o None."""
        alcance = self.alcance
        if hr.coincide(ruta, alcance["rutas_prohibidas"]):
            return f"Contrato: {ruta} está entre las rutas que no puedes tocar."
        if not hr.coincide(ruta, alcance["rutas_permitidas"]):
            return (f"Contrato: {ruta} no está entre las rutas que puedes cambiar ("
                    + ", ".join(alcance["rutas_permitidas"]) + ").")
        return super().regla_rota(nombre, ruta)

## 4. La retroalimentación y la puerta de verificación del capítulo 5

In [ ]:
MAX_FALLOS = 3   # fallos que se resumen para el agente


def resumir_fallos(salida, carpeta):
    """Los fallos de una ejecución de unittest, uno por prueba: qué prueba falla, las
    líneas del repositorio por las que pasa el error (sin las de Python) y la última
    línea, la del error. Las rutas se dejan relativas a la copia del repositorio."""
    fallos = []
    prefijo = str(carpeta) + "\\" if "\\" in str(carpeta) else str(carpeta) + "/"
    for bloque in re.split(r"^={50,}$", salida, flags=re.M)[1:]:
        cabecera = re.search(r"^(FAIL|ERROR): (\S+)", bloque, re.M)
        if not cabecera:
            continue
        lineas = bloque.split("-" * 70)[1] if "-" * 70 in bloque else bloque
        donde = []
        for m in re.finditer(r'File "([^"]+)", line (\d+), in (\S+)\n\s+(.+)', lineas):
            if m.group(1).startswith(prefijo):
                ruta = m.group(1)[len(prefijo):].replace("\\", "/")
                donde.append(f"{ruta}:{m.group(2)} ({m.group(3)}): {m.group(4).strip()}")
        error = [l for l in lineas.strip().splitlines() if l.strip()][-1:]
        fallos.append({"prueba": f"{cabecera.group(1)} {cabecera.group(2)}",
                       "donde": donde[-3:], "error": error[0].strip() if error else ""})
    return fallos[:MAX_FALLOS]


class HerramientasConRetro(HerramientasConContrato):
    """Las herramientas con el contrato y, además, retroalimentación: después de cada
    edición que sale bien se ejecutan las pruebas y el resultado de la edición lleva
    un resumen de los fallos. ejecutar_pruebas devuelve el mismo resumen."""

    def pruebas(self):
        """Las pruebas del repositorio, resumidas para el agente."""
        resultado = self.herramientas.ejecutar_pruebas()
        resumen = {"todas_pasan": resultado["todas_pasan"],
                   "resultado": resultado["resultado"]}
        if not resultado["todas_pasan"]:
            resumen["fallos"] = resumir_fallos(resultado["salida"],
                                               self.herramientas.espacio.carpeta)
        return resumen

    def llamar(self, nombre, argumentos):
        """Como HerramientasConContrato.llamar, con las pruebas tras cada edición."""
        if nombre == "ejecutar_pruebas":
            resultado = self.pruebas()
            self.registro.append({"herramienta": nombre, "argumentos": {}, "ok": True,
                                  "segundos": 0, "resultado": resultado})
            return resultado
        resultado = super().llamar(nombre, argumentos)
        if nombre in ("editar", "crear_archivo") and "error" not in resultado:
            resultado = dict(resultado, pruebas_tras_editar=self.pruebas())
        return resultado


def codigo_sin_docstrings(texto):
    """El árbol sintáctico de un programa, como texto, sin sus docstrings. Los
    comentarios no llegan al árbol, así que dos versiones que solo se diferencian en
    comentarios o docstrings dan el mismo resultado. None si no se puede analizar."""
    try:
        arbol = ast.parse(texto)
    except SyntaxError:
        return None
    for nodo in ast.walk(arbol):
        cuerpo = getattr(nodo, "body", None)
        if (isinstance(cuerpo, list) and cuerpo and isinstance(cuerpo[0], ast.Expr)
                and isinstance(cuerpo[0].value, ast.Constant)
                and isinstance(cuerpo[0].value.value, str)):
            nodo.body = cuerpo[1:] or [ast.Pass()]
    return ast.dump(arbol)


def cambia_el_codigo(espacio, ruta):
    """True si el archivo ha cambiado en algo más que comentarios y docstrings."""
    original, nuevo = espacio.origen / ruta, espacio.ruta(ruta)
    if not ruta.endswith(".py") or not original.exists() or not nuevo.exists():
        return True
    antes = codigo_sin_docstrings(original.read_text(encoding="utf-8"))
    despues = codigo_sin_docstrings(nuevo.read_text(encoding="utf-8"))
    return antes is None or despues is None or antes != despues


def puerta_de_verificacion(espacio, alcance):
    """Lo que falta para dar la tarea por hecha, mirando el repositorio y no lo que dice
    el agente: cambios de código que no sean solo comentarios, una prueba nueva que
    falle con el código original (si el contrato deja tocar tests/) y todas las
    pruebas en verde. Devuelve una lista de frases; vacía si se puede aceptar."""
    cambios = hr.diferencias(espacio.origen, espacio.carpeta)
    lineas = lineas_del_diff(cambios["diff"])
    falta = []
    codigo = [r for r in cambios["cambiados"]
              if not r.startswith(("tests/", "docs/", "datos/"))]
    if not any(cambia_el_codigo(espacio, r) for r in codigo):
        falta.append("no hay cambios de código en el programa (los comentarios no cuentan)")
    pruebas_nuevas = [r for r, l in lineas.items() if r.startswith("tests/")
                      and any(re.search(r"def test_|assert", x) for x in l["+"])]
    if not prohibe_pruebas(alcance):
        if not pruebas_nuevas:
            falta.append("no hay ninguna prueba nueva en tests/")
        else:
            with hr.Espacio() as original:      # las pruebas nuevas, con el código de antes
                for ruta in pruebas_nuevas:
                    destino = original.ruta(ruta)
                    destino.write_text(espacio.ruta(ruta).read_text(encoding="utf-8"),
                                       encoding="utf-8")
                antes = hr.Herramientas(original).ejecutar_pruebas()
            if antes["todas_pasan"]:
                falta.append("tus pruebas nuevas también pasan con el código original, así "
                             "que no comprueban el cambio que pide la petición")
    ahora = hr.Herramientas(espacio).ejecutar_pruebas()
    if not ahora["todas_pasan"]:
        fallos = resumir_fallos(ahora["salida"], espacio.carpeta)
        falta.append("las pruebas no pasan: "
                     + "; ".join(f"{f['prueba']} → {f['error']}" for f in fallos))
    return falta

## 5. El revisor del capítulo 6 y el marco y el plan del capítulo 7

In [ ]:
MAX_PASOS_REVISOR = 10   # llamadas al modelo en cada revisión


MAX_REVISIONES = 2       # veces que el revisor puede devolver el trabajo


ESCALADO = ("NO TERMINADO: el revisor ha rechazado el cambio dos veces; queda "
            "pendiente de que lo revise una persona.")


SOLO_LECTURA = ("listar_archivos", "leer_archivo", "buscar", "ejecutar_pruebas")


SISTEMA_REVISOR = (
    "Eres el revisor de cambios del repositorio de Ciclos Norte, una tienda de "
    "bicicletas de Oviedo. Otro agente ha hecho un cambio para resolver una petición y "
    "dice que ha terminado. Tu trabajo es comprobar si el cambio resuelve de verdad la "
    "petición. Puedes listar, leer y buscar archivos y ejecutar las pruebas, pero no "
    "puedes cambiar nada. Usa una herramienta cada vez. Cuando hayas terminado, "
    "responde sin llamar a ninguna herramienta, solo con un JSON.")


PIDE_LIBRE = """Revisa este cambio.

Petición:
{peticion}

Cambio (diff):
{diff}

Lo que dice el autor:
{respuesta}

Responde al final con este JSON:
{{"aprobado": true o false, "comentarios": ["…"]}}"""


PIDE_RUBRICA = """Revisa este cambio criterio por criterio.

Petición:
{peticion}

Criterios de aceptación:
{criterios}

Cambio (diff):
{diff}

Lo que dice el autor:
{respuesta}

Para cada criterio, busca en el código (no en lo que dice el autor) si se cumple, y da
la evidencia: archivo y línea, o la prueba que lo comprueba. Mira también si el cambio
toca algo que la petición no pide. Responde al final con este JSON:
{{"criterios": [{{"id": "C1", "cumple": true o false, "evidencia": "…"}}],
  "fuera_de_lo_pedido": ["…"], "aprobado": true o false}}
Aprueba solo si se cumplen todos los criterios."""


class HerramientasDeLectura:
    """Las herramientas del revisor: las de siempre, pero solo las que no cambian
    nada. Cualquier otra se rechaza."""

    def __init__(self, herramientas):
        self.herramientas = herramientas
        self.registro = []

    def llamar(self, nombre, argumentos):
        """Como Herramientas.llamar, solo para las herramientas de lectura."""
        if nombre not in SOLO_LECTURA:
            resultado = {"error": f"El revisor no puede usar «{nombre}»: solo "
                                  + ", ".join(SOLO_LECTURA) + "."}
        else:
            resultado = self.herramientas.llamar(nombre, argumentos)
        self.registro.append({"herramienta": nombre, "ok": "error" not in resultado})
        return resultado


def leer_veredicto(respuesta):
    """El último objeto JSON de la respuesta del revisor, o None si no hay ninguno que
    tenga la clave «aprobado»."""
    for trozo in reversed(re.findall(r"\{.*\}", respuesta, re.S)):
        try:
            veredicto = json.loads(trozo, strict=False)
        except ValueError:
            continue
        if isinstance(veredicto, dict) and "aprobado" in veredicto:
            return veredicto
    return None


def revisar(tok, modelo, tarea, alcance, espacio, respuesta_autor, rubrica, ver=False):
    """Una revisión: un agente aparte, con su propia conversación y herramientas de
    solo lectura sobre la copia del repositorio, que devuelve su veredicto. Sin
    veredicto legible, se aprueba (el revisor no bloquea por fallar él)."""
    cambios = hr.diferencias(espacio.origen, espacio.carpeta)
    criterios = "\n".join(f"C{i}. {c}" for i, c in
                          enumerate(alcance["criterios_aceptacion"], 1))
    plantilla_peticion = PIDE_RUBRICA if rubrica else PIDE_LIBRE
    mensajes = [{"role": "system", "content": SISTEMA_REVISOR},
                {"role": "user", "content": plantilla_peticion.format(
                    peticion=tarea["peticion"], criterios=criterios,
                    diff=cambios["diff"][:3000], respuesta=respuesta_autor[:800])}]
    herramientas = HerramientasDeLectura(hr.Herramientas(espacio))
    veredicto, pasos = None, 0
    for pasos in range(1, MAX_PASOS_REVISOR + 1):
        texto, recortados = recortar_conversacion(tok, mensajes)
        if recortados is None:
            break
        respuesta, _, _ = generar(tok, modelo, texto)
        llamadas = leer_llamadas(respuesta)
        if not llamadas:
            veredicto = leer_veredicto(respuesta)
            break
        mensajes.append({"role": "assistant", "content": respuesta.split("<tool_call>")[0],
                         "tool_calls": [{"type": "function", "function": l}
                                        for l in llamadas if l["name"]]})
        for llamada in llamadas:
            resultado = (herramientas.llamar(llamada["name"], llamada["arguments"])
                         if llamada["name"] else {"error": "Llamada no válida."})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": json.dumps(resultado, ensure_ascii=False)})
    if ver:
        print(f"    REVISOR ({pasos} pasos): {json.dumps(veredicto, ensure_ascii=False)[:300]}")
    return {"aprobado": True if veredicto is None else bool(veredicto.get("aprobado")),
            "veredicto": veredicto, "pasos": pasos,
            "llamadas": len(herramientas.registro)}


def comentarios_para_el_autor(veredicto):
    """Lo que el revisor objeta, en frases para el autor."""
    if not veredicto:
        return "el revisor no aprueba el cambio"
    frases = [f"{c.get('id', '?')} no se cumple: {c.get('evidencia', '')}"
              for c in veredicto.get("criterios", []) if not c.get("cumple", True)]
    frases += [f"fuera de lo pedido: {x}" for x in veredicto.get("fuera_de_lo_pedido", [])]
    frases += [str(x) for x in veredicto.get("comentarios", [])]
    return "; ".join(frases) or "el revisor no aprueba el cambio"

In [ ]:
MAX_PASOS_MARCO = 10   # llamadas al modelo para explorar y escribir el marco


PIDE_MARCO = """Antes de cambiar nada, entiende la tarea. Explora el repositorio con las
herramientas de lectura y escribe el marco de la tarea en JSON, sin nada más al final:
{{"objetivo": "una frase",
  "hechos": [{{"hecho": "lo que has comprobado", "evidencia": "archivo:línea"}}],
  "rutas": ["archivos que habrá que cambiar"],
  "incognitas": ["lo que no sabes y cómo lo averiguarías"]}}
Cada hecho debe citar un archivo y una línea que lo demuestren.

Petición:
{peticion}

{contrato}"""


PIDE_PLAN = """Ahora escribe el plan para resolverla, en JSON, sin nada más al final:
{{"pasos": [{{"paso": "qué hacer", "archivo": "dónde", "comprobacion": "qué prueba lo demuestra"}}]}}
Cada paso debe decir cómo se comprobará que está bien hecho."""


def leer_json(respuesta, clave):
    """El último objeto JSON de una respuesta que tenga la clave dada, o None."""
    for trozo in reversed(re.findall(r"\{.*\}", respuesta, re.S)):
        try:
            datos = json.loads(trozo, strict=False)
        except ValueError:
            continue
        if isinstance(datos, dict) and clave in datos:
            return datos
    return None


def comprobar_evidencias(marco, espacio):
    """Los hechos del marco cuya evidencia no existe: un archivo que no está o una
    línea que no tiene."""
    malas = []
    for hecho in marco.get("hechos", []):
        m = re.match(r"\s*([\w./-]+\.\w+)(?::(\d+))?", str(hecho.get("evidencia", "")))
        archivo = espacio.ruta(m.group(1)) if m else None
        if not m or not archivo.is_file():
            malas.append(f"«{hecho.get('hecho', '')}»: no existe {hecho.get('evidencia')}")
        elif m.group(2) and int(m.group(2)) > len(
                archivo.read_text(encoding="utf-8").splitlines()):
            malas.append(f"«{hecho.get('hecho', '')}»: {m.group(1)} no tiene la línea "
                         f"{m.group(2)}")
    return malas


def conversar(tok, modelo, mensajes, herramientas, max_pasos, clave, ver=False):
    """Una conversación con herramientas de lectura que acaba con un JSON con «clave».
    Devuelve el JSON (o None) y los pasos."""
    for pasos in range(1, max_pasos + 1):
        texto, recortados = recortar_conversacion(tok, mensajes)
        if recortados is None:
            return None, pasos
        respuesta, _, _ = generar(tok, modelo, texto)
        llamadas = leer_llamadas(respuesta)
        if not llamadas:
            mensajes.append({"role": "assistant", "content": respuesta})
            return leer_json(respuesta, clave), pasos
        mensajes.append({"role": "assistant", "content": respuesta.split("<tool_call>")[0],
                         "tool_calls": [{"type": "function", "function": l}
                                        for l in llamadas if l["name"]]})
        for llamada in llamadas:
            resultado = (herramientas.llamar(llamada["name"], llamada["arguments"])
                         if llamada["name"] else {"error": "Llamada no válida."})
            if ver:
                print(f"  [marco {pasos}] {llamada['name']}"
                      f"({json.dumps(llamada['arguments'], ensure_ascii=False)[:80]})")
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": json.dumps(resultado, ensure_ascii=False)})
    return None, max_pasos


def enmarcar(tok, modelo, tarea, espacio, con_plan, ver=False):
    """La fase previa: el agente explora sin cambiar nada y escribe el marco de la
    tarea; el programa comprueba las evidencias y, si alguna no existe, se lo devuelve
    una vez. Con «con_plan», escribe además un plan con una comprobación por paso.
    Devuelve el texto
    que se añade a la petición y lo que ha pasado."""
    alcance = tarea["alcance"]
    mensajes = [{"role": "system", "content": SISTEMA + "\n\n" + AGENTS_REGLAS},
                {"role": "user", "content": PIDE_MARCO.format(
                    peticion=tarea["peticion"], contrato=contrato_texto(alcance))}]
    lectura = HerramientasDeLectura(hr.Herramientas(espacio))
    marco, pasos = conversar(tok, modelo, mensajes, lectura, MAX_PASOS_MARCO,
                             "objetivo", ver)
    malas = comprobar_evidencias(marco, espacio) if marco else []
    if malas:
        mensajes.append({"role": "user", "content":
                         "Estas evidencias no existen: " + "; ".join(malas)
                         + ". Corrige el marco y vuelve a escribirlo en JSON."})
        corregido, mas = conversar(tok, modelo, mensajes, lectura, 4, "objetivo", ver)
        pasos += mas
        marco = corregido or marco
        malas = comprobar_evidencias(marco, espacio) if marco else malas
    plan = None
    if con_plan and marco:
        mensajes.append({"role": "user", "content": PIDE_PLAN})
        plan, mas = conversar(tok, modelo, mensajes, lectura, 2, "pasos", ver)
        pasos += mas
    partes = []
    if marco:
        partes.append("Marco de la tarea (escrito antes de empezar):\n"
                      + json.dumps(marco, ensure_ascii=False, indent=1)[:2500])
    if plan:
        partes.append("Plan:\n" + json.dumps(plan, ensure_ascii=False, indent=1)[:1500])
    return "\n\n".join(partes), {"marco": marco, "plan": plan, "pasos": pasos,
                                 "evidencias_malas": malas}

## 6. Un agente que trabaja por turnos

`trabajar` es un generador: devuelve el control después de cada llamada al modelo.

In [ ]:
def trabajar(tok, modelo, tarea, espacio, marco=False, plan=False, revisor=None,
             max_pasos=MAX_PASOS, ver=False):
    """El agente del capítulo 7 (contrato, retroalimentación, puerta de verificación,
    revisor y, si se pide, marco y plan) sobre la copia del repositorio que se le da.
    Es un generador: devuelve el control después de cada llamada al modelo, para que
    varios agentes puedan trabajar por turnos. Al acabar devuelve lo que ha pasado."""
    alcance = tarea["alcance"]
    peticion = tarea["peticion"] + "\n\n" + contrato_texto(alcance)
    fase = {"marco": None, "plan": None, "pasos": 0, "evidencias_malas": []}
    if marco:
        previo, fase = enmarcar(tok, modelo, tarea, espacio, con_plan=plan, ver=ver)
        if previo:
            peticion += "\n\n" + previo
        yield
    mensajes = [{"role": "system", "content": SISTEMA + "\n\n" + AGENTS_REGLAS},
                {"role": "user", "content": peticion}]
    final, pasos, entrada, salida, rechazos = None, 0, 0, 0, 0
    revisiones = []
    base = hr.Herramientas(espacio)
    herramientas = HerramientasConRetro(base, tarea, alcance)
    for pasos in range(1, max_pasos + 1):
        texto, recortados = recortar_conversacion(tok, mensajes)
        if recortados is None:
            break
        respuesta, n_entrada, n_salida = generar(tok, modelo, texto)
        entrada, salida = entrada + n_entrada, salida + n_salida
        yield   # aquí pueden trabajar los demás agentes
        llamadas = leer_llamadas(respuesta)
        if not llamadas:
            mensajes.append({"role": "assistant", "content": respuesta})
            if ver:
                print(f"[{tarea['id']} {pasos}] {respuesta}")
            if dice_terminado(respuesta) and rechazos < MAX_RECHAZOS:
                cambios = hr.diferencias(espacio.origen, espacio.carpeta)
                rotas = comprobar_reglas(tarea, base.registro, cambios, respuesta,
                                         exigir_prueba=not prohibe_pruebas(alcance),
                                         permite_docs=herramientas.permite_docs())
                rotas.update(comprobar_contrato(alcance, cambios, respuesta))
                rotas.pop("R7", None)
                falta = puerta_de_verificacion(espacio, alcance)
                if falta:
                    rotas["Verificación"] = "; ".join(falta)
                if rotas:
                    rechazos += 1
                    mensajes.append({"role": "user", "content":
                                     "Todavía no está terminado: "
                                     + "; ".join(f"{r}: {t}" for r, t in
                                                 sorted(rotas.items()))
                                     + ". Sigue trabajando."})
                    continue
            if revisor and dice_terminado(respuesta):
                if len(revisiones) == MAX_REVISIONES:   # ya lo rechazó dos veces
                    final = ESCALADO
                    break
                revision = revisar(tok, modelo, tarea, alcance, espacio, respuesta,
                                   rubrica=revisor == "rúbrica", ver=ver)
                revisiones.append(revision)
                if not revision["aprobado"]:
                    mensajes.append({"role": "user", "content":
                                     "El revisor no aprueba el cambio: "
                                     + comentarios_para_el_autor(revision["veredicto"])
                                     + ". Corrígelo y vuelve a responder TERMINADO."})
                    continue
            final = respuesta
            break
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        mensajes.append({"role": "assistant", "content": pensamiento,
                         "tool_calls": [{"type": "function", "function": l}
                                        for l in llamadas if l["name"]]})
        for llamada in llamadas:
            if llamada["name"] is None:
                resultado = {"error": "No se entiende la llamada a la herramienta."}
            else:
                resultado = herramientas.llamar(llamada["name"], llamada["arguments"])
            if ver:
                print(f"[{tarea['id']} {pasos}] {llamada['name']}"
                      f"({json.dumps(llamada['arguments'], ensure_ascii=False)[:90]})"
                      f" -> {json.dumps(resultado, ensure_ascii=False)[:110]}")
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": json.dumps(resultado, ensure_ascii=False)})
    return {"final": final, "pasos": pasos, "entrada": entrada, "salida": salida,
            "rechazos": rechazos, "revisiones": revisiones, "fase": fase,
            "registro": base.registro}


def hasta_el_final(generador):
    """Hace avanzar un generador hasta que acaba y devuelve su valor final."""
    while True:
        try:
            next(generador)
        except StopIteration as fin:
            return fin.value


def leer_cambios(espacio):
    """Los archivos que han cambiado en una copia, con su texto final (None si se ha
    borrado)."""
    cambios = hr.diferencias(espacio.origen, espacio.carpeta)
    return {r: None if r in cambios["borrados"] else espacio.ruta(r).read_text(
                encoding="utf-8") for r in cambios["cambiados"]}


def agente(tok, modelo, tarea, origen=None, ver=False, **opciones):
    """Un agente en su propia copia del repositorio: la del original o, con «origen»,
    la de otra carpeta, por ejemplo lo que ya se ha integrado. Devuelve la fila de
    resultados, con los archivos que ha cambiado y cómo han quedado."""
    inicio = time.time()
    with (hr.Espacio(origen) if origen else hr.Espacio()) as espacio:
        hecho = hasta_el_final(trabajar(tok, modelo, tarea, espacio, ver=ver, **opciones))
        evaluacion = hr.evaluar_intento(espacio, tarea)
        cambios = leer_cambios(espacio)
    terminado = hecho["final"] is not None and dice_terminado(hecho["final"])
    return {"tarea": tarea["id"], "resuelta": evaluacion["resuelta"],
            "dice_terminado": terminado,
            "terminado_falso": terminado and not evaluacion["resuelta"],
            "visibles": evaluacion["visibles"], "ocultas": evaluacion["ocultas"],
            "fuera_de_alcance": evaluacion["fuera_de_alcance"],
            "rechazos": hecho["rechazos"], "revisiones": len(hecho["revisiones"]),
            "pasos": hecho["pasos"] + hecho["fase"]["pasos"],
            "tokens_entrada": hecho["entrada"],
            "segundos": round(time.time() - inicio, 1),
            "respuesta": hecho["final"] or "(sin respuesta final)", "cambios": cambios}

## 7. Integrar

Se puede probar sin modelo, con las soluciones de referencia: las de T15 y T25 chocan en las condiciones generales; las de T30 y T20 se fusionan sin conflictos, pero rompen T20.

In [ ]:
def fusionar_archivo(base, nuestro, suyo):
    """Fusión a tres bandas de un archivo con git merge-file: lo que había cuando la
    tarea empezó (base), lo integrado hasta ahora (nuestro) y lo que ha hecho la tarea
    (suyo). Devuelve el texto fusionado y si ha habido conflicto."""
    with tempfile.TemporaryDirectory() as carpeta:
        rutas = []
        for nombre, texto in (("nuestro", nuestro), ("base", base), ("suyo", suyo)):
            ruta = Path(carpeta) / nombre
            ruta.write_bytes(texto.encode("utf-8"))   # en bytes: los saltos de línea
            rutas.append(str(ruta))                   # no cambian en Windows
        r = subprocess.run(["git", "merge-file", "-p", "-L", "integrado", "-L", "original",
                            "-L", "tarea", *rutas], capture_output=True)
    return r.stdout.decode("utf-8"), r.returncode != 0


def integrar(integrado, base, cambios, fusionar=True):
    """Lleva los cambios de una tarea a la copia integrada. Sin fusionar, copia los
    archivos tal cual y gana el último. Fusionando, cada archivo que ya existía se
    fusiona a tres bandas con lo integrado. Devuelve los archivos con conflicto."""
    conflictos = []
    for relativa, texto in cambios.items():
        destino, original = integrado.ruta(relativa), Path(base) / relativa
        if texto is None:
            destino.unlink(missing_ok=True)
            continue
        if fusionar and original.exists() and destino.exists():
            texto, conflicto = fusionar_archivo(original.read_text(encoding="utf-8"),
                                                destino.read_text(encoding="utf-8"), texto)
            if conflicto:
                conflictos.append(relativa)
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.write_bytes(texto.encode("utf-8"))
    return conflictos


def foto(integrado, rutas):
    """El texto actual de unos archivos de la copia integrada, para poder volver atrás."""
    return {r: integrado.ruta(r).read_text(encoding="utf-8")
            if integrado.ruta(r).exists() else None for r in rutas}


def verde(espacio):
    """Si pasan todas las pruebas visibles de una copia."""
    return hr.Herramientas(espacio).ejecutar_pruebas()["todas_pasan"]

In [ ]:
def con_la_solucion(identificador):
    """Los cambios de la solución de referencia de una tarea, sin modelo."""
    with hr.Espacio() as espacio:
        hr.aplicar_solucion(hr.Herramientas(espacio), hr.leer_tarea(identificador)["carpeta"] / "solucion")
        return leer_cambios(espacio)

for pareja in (("T15", "T25"), ("T30", "T20")):
    with hr.Espacio() as integrado:
        conflictos = [c for i in pareja for c in integrar(integrado, integrado.origen, con_la_solucion(i))]
        ocultas = {i: hr.pasar_ocultas(integrado.carpeta, hr.leer_tarea(i))["pasan"] for i in pareja}
    print(pareja, "conflictos:", conflictos, "· pruebas ocultas:", ocultas)

## 8. Cuatro formas de repartir un lote

Por defecto se prueban dos tareas del lote de los pedidos, cada una en su copia, y se juntan de dos formas: copiando los cambios uno detrás de otro y con el integrador. Con `COMPLETO = True`, el lote entero con las cuatro formas. Después, los resultados guardados de las tres ejecuciones completas.

In [ ]:
LOTES = {"pedidos": ["T22", "T28", "T29"], "facturas": ["T04", "T26", "T12"],
         "existencias": ["T24", "T23", "T05"], "avisos": ["T30", "T11", "T20"],
         "envíos": ["T15", "T25", "T17"]}


AGENTE = {"marco": True, "plan": True, "revisor": "rúbrica"}   # el del capítulo 7


def resultado_del_lote(espacio, tareas, pasos, ruta, **otros):
    """Cómo queda el lote: las pruebas visibles de la copia final, las ocultas de cada
    tarea, los pasos de todos los agentes y los de la ruta más larga."""
    ocultas = {t["id"]: hr.pasar_ocultas(espacio.carpeta, t)["pasan"] for t in tareas}
    return {"visibles": verde(espacio), "ocultas": ocultas,
            "en_pie": sum(ocultas.values()), "pasos": pasos, "ruta": ruta, **otros}


def misma_copia(tok, modelo, tareas, ver=False):
    """Las tareas a la vez, por turnos, en la misma copia del repositorio."""
    with hr.Espacio() as espacio:
        agentes = {t["id"]: trabajar(tok, modelo, t, espacio, ver=ver, **AGENTE)
                   for t in tareas}
        hechos = {}
        while agentes:
            for identificador in list(agentes):
                try:
                    next(agentes[identificador])
                except StopIteration as fin:
                    hechos[identificador] = fin.value
                    del agentes[identificador]
        pasos = {i: h["pasos"] + h["fase"]["pasos"] for i, h in hechos.items()}
        return resultado_del_lote(espacio, tareas, sum(pasos.values()),
                                  max(pasos.values()))


def por_separado(tok, modelo, tareas, ver=False):
    """Cada tarea en su propia copia del original, todas desde el mismo punto."""
    return [agente(tok, modelo, t, ver=ver, **AGENTE) for t in tareas]


def ultimo_gana(tareas, filas):
    """Los cambios de cada copia, copiados uno detrás de otro sobre el original."""
    with hr.Espacio() as integrado:
        for fila in filas:
            integrar(integrado, integrado.origen, fila["cambios"], fusionar=False)
        return resultado_del_lote(integrado, tareas, sum(f["pasos"] for f in filas),
                                  max(f["pasos"] for f in filas))


def con_integrador(tok, modelo, tareas, filas, ver=False):
    """Cada tarea en su copia, y un integrador que fusiona sus cambios uno a uno. Solo
    entra lo que deja las pruebas en verde: si una fusión tiene conflictos o rompe
    alguna prueba, la deshace y pide la tarea otra vez, ahora sobre lo ya integrado; si
    vuelve a fallar, la deja para una persona."""
    pasos, ruta = sum(f["pasos"] for f in filas), max(f["pasos"] for f in filas)
    conflictos, rehechas, fuera = 0, [], []
    with hr.Espacio() as integrado:
        for tarea, fila in zip(tareas, filas):
            antes = foto(integrado, fila["cambios"])
            choques = integrar(integrado, integrado.origen, fila["cambios"])
            conflictos += len(choques)
            if not choques and verde(integrado):
                continue
            integrar(integrado, integrado.carpeta, antes, fusionar=False)   # deshacer
            otra = agente(tok, modelo, tarea, origen=integrado.carpeta, ver=ver, **AGENTE)
            pasos, ruta = pasos + otra["pasos"], ruta + otra["pasos"]
            rehechas.append(tarea["id"])
            antes = foto(integrado, otra["cambios"])
            integrar(integrado, integrado.carpeta, otra["cambios"])
            if not verde(integrado):
                integrar(integrado, integrado.carpeta, antes, fusionar=False)
                fuera.append(tarea["id"])
        return resultado_del_lote(integrado, tareas, pasos, ruta, conflictos=conflictos,
                                  rehechas=rehechas, fuera=fuera)


def comparten(a, b):
    """Si dos tareas pueden tocar el mismo archivo, según las rutas de sus contratos."""
    return bool(set(a["alcance"]["rutas_permitidas"])
                & set(b["alcance"]["rutas_permitidas"]))


def en_oleadas(tok, modelo, tareas, filas, ver=False):
    """Rutas exclusivas: dos tareas que comparten archivos no trabajan a la vez. Se
    reparten en oleadas; las de una oleada trabajan cada una en su copia de lo ya
    integrado, y al acabar se integran todas. La primera oleada sale del original, así
    que sus filas son las de por_separado."""
    oleadas = []
    for tarea in tareas:
        libre = next((o for o in oleadas if not any(comparten(tarea, t) for t in o)), None)
        if libre is None:
            oleadas.append([tarea])
        else:
            libre.append(tarea)
    hechas = {f["tarea"]: f for f in filas}
    pasos = ruta = 0
    with hr.Espacio() as integrado:
        for numero, oleada in enumerate(oleadas):
            if numero:
                hechas.update({t["id"]: agente(tok, modelo, t, origen=integrado.carpeta,
                                               ver=ver, **AGENTE) for t in oleada})
            base = integrado.origen if numero == 0 else integrado.carpeta
            for tarea in oleada:
                integrar(integrado, base, hechas[tarea["id"]]["cambios"])
            pasos += sum(hechas[t["id"]]["pasos"] for t in oleada)
            ruta += max(hechas[t["id"]]["pasos"] for t in oleada)
        return resultado_del_lote(integrado, tareas, pasos, ruta, oleadas=len(oleadas))


def probar(tok, modelo, semilla=None, lotes=LOTES, ver=False):
    """Los lotes con las cuatro formas de repartirlos. Sin semilla, con decodificación
    voraz; con semilla, con muestreo, y cada parte con su propia semilla."""
    decodificacion.clear()
    decodificacion.update(VORAZ if semilla is None else MUESTREO)
    resultados = {}
    for nombre, identificadores in lotes.items():
        tareas = [hr.leer_tarea(i) for i in identificadores]
        numero = list(LOTES).index(nombre) if nombre in LOTES else 0

        def sembrar(parte):
            if semilla is not None:
                torch.manual_seed(1000 * semilla + 10 * numero + parte)

        sembrar(0)
        filas = por_separado(tok, modelo, tareas, ver)
        sembrar(1)
        lote = {"por separado": [{k: v for k, v in f.items() if k != "cambios"}
                                 for f in filas],
                "misma copia": misma_copia(tok, modelo, tareas, ver),
                "último gana": ultimo_gana(tareas, filas)}
        sembrar(2)
        lote["con integrador"] = con_integrador(tok, modelo, tareas, filas, ver)
        sembrar(3)
        lote["en oleadas"] = en_oleadas(tok, modelo, tareas, filas, ver)
        resultados[nombre] = lote
        print(f"  {nombre:12s} " + "  ".join(
            f"{k}: {v['en_pie']}/3" for k, v in lote.items() if k != "por separado")
            + f"  (en su copia: {sum(f['resuelta'] for f in filas)}/3)", flush=True)
    return resultados

In [ ]:
# Con COMPLETO = False se prueban dos tareas con dos formas (unos 50 minutos en la T4 de Colab);
# con True, el lote entero con las cuatro formas (unas tres horas en la T4).
COMPLETO = False
if COMPLETO:
    resultados = probar(tok, modelo, semilla=1, lotes={"pedidos": LOTES["pedidos"]})["pedidos"]
else:
    decodificacion.clear()
    decodificacion.update(MUESTREO)
    torch.manual_seed(1)
    tareas = [hr.leer_tarea(i) for i in ("T28", "T29")]
    filas = por_separado(tok, modelo, tareas)
    resultados = {"último gana": ultimo_gana(tareas, filas),
                  "con integrador": con_integrador(tok, modelo, tareas, filas)}
for forma, r in resultados.items():
    if forma != "por separado":
        print(f"{forma:15s} en pie {r['en_pie']}/{len(r['ocultas'])}  pruebas en verde {r['visibles']}")

In [ ]:
guardados = json.loads((RAIZ / "datos" / "resultados" / "capitulo_8" / "resultados_repeticiones.json").read_text(encoding="utf-8"))
for clave, lote in guardados.items():
    for forma, r in lote.items():
        if forma != "por separado":
            print(f"{clave:22s} {forma:15s} en pie {r['en_pie']}/3")

## Ejercicios

Los archivos de resultados que citan los ejercicios están en `datos/resultados/capitulo_8/`, que `preparar` descarga con los datos.

1. ¿Qué diferencia hay entre un conflicto textual y uno semántico? Pon un ejemplo de cada uno con las tareas de Ciclos Norte. *(Sin GPU.)*

2. Con `resultados_repeticiones.json`, busca en el lote de los pedidos qué tareas se resuelven en su copia y no siguen en pie con «gana la última». ¿Qué archivo las borró? *(Sin GPU.)*

3. Aplica las soluciones de referencia de T15 y T25, que están en `datos/tareas/Txx/solucion`, sobre dos copias del repositorio y fusiona las condiciones generales con `fusionar_archivo`. Resuelve el conflicto a mano. ¿Qué tendría que saber un agente para resolverlo él? *(Sin GPU.)*

4. Haz lo mismo con las soluciones de T30 y T20, que se fusionan sin conflictos, y pasa las pruebas ocultas de las dos. Escribe una prueba visible que habría detectado el conflicto semántico. *(Sin GPU.)*

5. Cuenta en los resultados cuántas tareas pidió otra vez el integrador, cuántas salieron bien a la segunda y cuántas quedaron fuera. Para las que quedaron fuera, ¿qué le contarías a la persona que las recibe? *(Sin GPU.)*

6. Si cada llamada al modelo tarda tres segundos y cada agente tiene su propia GPU, ¿cuánto tarda cada forma de repartir según su ruta? ¿Y con una sola GPU? ¿Cuál elegirías en cada caso? *(Sin GPU.)*

7. Combina las oleadas con la comprobación del integrador: una tarea detrás de otra sobre lo ya integrado, pero solo entra lo que deja las pruebas en verde. Pruébalo con el lote de los pedidos. Los resultados de una ejecución están en `resultados_ejercicios.json`. *(GPU, unos 50 minutos, o sin GPU con los resultados guardados.)*

8. En la misma copia, los agentes trabajan por turnos estrictos. Describe qué pasaría si dos agentes leyeran `pedidos.py`, pensaran a la vez y escribieran uno detrás de otro. ¿Qué herramienta del capítulo 2 lo evitaría en parte? *(Sin GPU.)*

9. `comparten` decide que dos tareas chocan si sus contratos permiten algún archivo común. Propón una forma más fina, por funciones en lugar de archivos, y di qué tareas de los cinco lotes podrían entonces trabajar a la vez. *(Sin GPU.)*

10. **Proyecto.** En un repositorio tuyo, da tres tareas a la vez a un agente de programación, cada una en su propio `git worktree`, y intégralas en una rama con integración continua. Anota cuántas entran a la primera, cuántas chocan y cuánto tiempo te cuesta resolver cada choque.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [git merge-file](https://git-scm.com/docs/git-merge-file). [Continuous Integration](https://martinfowler.com/articles/continuousIntegration.html) (Martin Fowler). La tienda Ciclos Norte, su programa y sus tareas son ficticios. Código y experimentos propios.